# Анализ удовлетворённости городскими услугами
Ноутбук читает 120 строк оценок (транспорт, медицина, образование) из файла `data.csv` и рассчитывает базовые статистики.

## Описание данных
Каждая строка содержит три числа через запятую: рейтинг транспорта, медицины и образования от 1 (минимум) до 5 (максимум). Файл создан заранее и включает не менее 100 наблюдений.

In [9]:

# Загрузка данных из файла без сторонних библиотек
# Каждая строка: transport, medicine, education (значения 1-5)
def load_responses(path):
    responses = []
    with open(path, 'r', encoding='utf-8') as file:
        for line in file:
            stripped = line.strip()
            if not stripped:
                continue
            parts = stripped.split(',')
            # Используем условие, чтобы убедиться, что в строке три числа
            if len(parts) != 3:
                # Пропустим некорректные строки, если появятся
                continue
            try:
                ratings = [int(value) for value in parts]
            except ValueError:
                continue
            # Проверяем, что значения попадают в ожидаемый диапазон 1-5
            if all(1 <= rating <= 5 for rating in ratings):
                responses.append(tuple(ratings))
    return responses

responses = load_responses('data.csv')
print(f'Всего строк в файле: {len(responses)}')


Всего строк в файле: 120


In [10]:

# Функции для расчёта статистик без внешних библиотек
def average(values):
    return sum(values) / len(values) if values else 0

def median(values):
    sorted_vals = sorted(values)
    n = len(sorted_vals)
    mid = n // 2
    if n % 2 == 1:
        return sorted_vals[mid]
    else:
        return (sorted_vals[mid - 1] + sorted_vals[mid]) / 2

def mode(values):
    counts = {}
    for value in values:
        counts[value] = counts.get(value, 0) + 1
    max_freq = max(counts.values())
    modes = [value for value, freq in counts.items() if freq == max_freq]
    # Возвращаем наименьшее модальное значение для стабильности
    return min(modes)

def variance(values):
    avg = average(values)
    return average([(value - avg) ** 2 for value in values])

def summarize_column(values):
    return {
        'count': len(values),
        'min': min(values),
        'max': max(values),
        'average': round(average(values), 2),
        'median': median(values),
        'mode': mode(values),
        'variance': round(variance(values), 2)
    }

transport_ratings = [resp[0] for resp in responses]
medicine_ratings = [resp[1] for resp in responses]
education_ratings = [resp[2] for resp in responses]

transport_stats = summarize_column(transport_ratings)
medicine_stats = summarize_column(medicine_ratings)
education_stats = summarize_column(education_ratings)

# Используем множество для подсчёта уникальных профилей оценок
unique_profiles = set(responses)
unique_profile_count = len(unique_profiles)

overall_average = round(average([average(resp) for resp in responses]), 2)
overall_median = median([average(resp) for resp in responses])

print('Число уникальных профилей оценок:', unique_profile_count)
print('Средняя общая удовлетворённость:', overall_average)
print('Медиана общей удовлетворённости:', overall_median)


Число уникальных профилей оценок: 76
Средняя общая удовлетворённость: 2.92
Медиана общей удовлетворённости: 3.0


In [11]:

# Словарь итоговых метрик для вывода и сохранения
results = {
    'transport': transport_stats,
    'medicine': medicine_stats,
    'education': education_stats,
    'unique_profiles': unique_profile_count,
    'overall_average': overall_average,
    'overall_median': overall_median,
}

# Условный оператор для оценки среднего уровня удовлетворённости
if overall_average >= 4.0:
    sentiment = 'Высокая удовлетворённость'
elif overall_average >= 3.0:
    sentiment = 'Средняя удовлетворённость'
else:
    sentiment = 'Низкая удовлетворённость'

results['sentiment'] = sentiment

# Подробный вывод на экран
print('--- Итоги по направлениям ---')
for category, stats in [('Транспорт', transport_stats), ('Медицина', medicine_stats), ('Образование', education_stats)]:
    print(category)
    for key, value in stats.items():
        print(f'  {key}: {value}')

print('Общая оценка населения:', sentiment)

# Сохранение результатов в файл
with open('results.txt', 'w', encoding='utf-8') as f:
    f.write('Анализ удовлетворённости городскими услугами\n')
    for category, stats in [('Транспорт', transport_stats), ('Медицина', medicine_stats), ('Образование', education_stats)]:
        f.write(category + '\n')
        for key, value in stats.items():
            f.write(f'  {key}: {value}\n')
        f.write('\n')
    f.write(f'Число уникальных профилей: {unique_profile_count}\n')
    f.write(f'Средняя общая удовлетворённость: {overall_average}\n')
    f.write(f'Медиана общей удовлетворённости: {overall_median}\n')
    f.write(f'Общая оценка населения: {sentiment}')

print('Результаты сохранены в results.txt')


--- Итоги по направлениям ---
Транспорт
  count: 120
  min: 1
  max: 5
  average: 2.79
  median: 2.0
  mode: 2
  variance: 2.05
Медицина
  count: 120
  min: 1
  max: 5
  average: 3.02
  median: 3.0
  mode: 2
  variance: 1.9
Образование
  count: 120
  min: 1
  max: 5
  average: 2.95
  median: 3.0
  mode: 1
  variance: 2.4
Общая оценка населения: Низкая удовлетворённость
Результаты сохранены в results.txt
